In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 概念章，CPU 即可
# ============================================================
# 本章在 1000 个 2D 点上训练一个隐藏维度 16 的 MLP，CPU 上 1 秒一轮。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Total VRAM     : {total_mem:.1f} GB')
else:
    print('未检测到 GPU —— 本章 CPU 完全够用，继续执行即可。')

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 必须是 cell 第一行，否则 Jupyter / VS Code 会把它当 line magic 报错。
# scikit-learn:  用 make_moons 生成线性不可分的玩具数据
# matplotlib:    画 loss 曲线和决策边界
# torch / torchvision 在 Colab 已预装，这里不动。
!pip install -q -U scikit-learn matplotlib

In [ ]:
# ============================================================
# Cell 2: 用 nn.Module 组织一个最简 MLP，并打印结构
# ============================================================
# 输入 2 维 → 隐藏 16 维 → 输出 1 维（二分类的 raw logit）
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, out_dim=1):
        # 必须先调父类构造，否则 nn.Module 内部状态（_parameters / _modules）不全
        super().__init__()
        # nn.Linear 内部存 weight shape (out, in)、bias shape (out,)
        # 默认 Kaiming uniform 初始化，无需我们手动初始化
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        # 中间层用 ReLU 引入非线性 —— 没它的话两层 Linear 等价于一层，模型彻底退化
        h = F.relu(self.fc1(x))
        # 最后一层不加激活，输出 raw logit；BCEWithLogitsLoss 内部会做 sigmoid
        y = self.fc2(h)
        return y

model = MLP()
print(model)

# 列出全部可训练参数的名字与 shape
print('\n=== named_parameters ===')
n_total = 0
for name, p in model.named_parameters():
    print(f'{name:12s} {tuple(p.shape)}  requires_grad={p.requires_grad}')
    n_total += p.numel()
print(f'\n总参数量 = {n_total}')

In [ ]:
# ============================================================
# Cell 3: 验证 nn.Linear 内部就是 y = x @ W.T + b
# ============================================================
# weight shape 是 (out, in) —— 这是 PyTorch 的约定，记反了 debug 会很痛苦。
import torch
import torch.nn as nn

torch.manual_seed(0)
linear = nn.Linear(in_features=3, out_features=2)

print('linear.weight.shape =', linear.weight.shape)   # (2, 3) ← (out, in)
print('linear.bias.shape   =', linear.bias.shape)     # (2,)

x = torch.randn(4, 3)              # batch=4, in_features=3
y_module  = linear(x)              # 走 nn.Linear
y_manual  = x @ linear.weight.T + linear.bias    # 手写公式

print('两种算法结果完全一致:', torch.allclose(y_module, y_manual))
print('y.shape =', y_module.shape)  # (4, 2)

In [ ]:
# ============================================================
# Cell 4: 准备数据 —— make_moons 两个交叉的半月形点云
# ============================================================
# 经典的"线性不可分"数据集：单层 Linear 分类器学不会，必须靠隐藏层 + ReLU。
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# n_samples=1000 个点；noise=0.2 让两类边界稍有交错
# random_state 固定 seed，保证每次跑得到一样的数据
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)

# numpy → tensor；BCEWithLogitsLoss 要求 target 是 float（不是 long）
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

print('X.shape =', X.shape, '  X.dtype =', X.dtype)
print('y.shape =', y.shape, '  y 取值=', torch.unique(y).tolist())

# 看一眼数据长啥样
plt.figure(figsize=(5, 4))
plt.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], s=10, label='class 0')
plt.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], s=10, label='class 1')
plt.title('make_moons (n=1000, noise=0.2)')
plt.legend()
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
# ============================================================
# Cell 5: 完整训练循环 —— forward → backward → step
# ============================================================
# 数据量小，全 batch 训练；真实大规模训练才需要 mini-batch (DataLoader)。
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)              # 让模型初始化、训练过程可复现

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

model = MLP()
# SGD 是最朴素的优化器，先用它跑通流程；2-16-1 这个 MLP 在 SGD lr=0.1 + 200 epoch
# 大约收敛到 acc ≈ 0.87（noise=0.2 下的瓶颈），换 AdamW 能到 0.95+，优化器细节留到 P04
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

losses = []
accs = []
for epoch in range(200):
    # ============= 三步走 =============
    # (a) 前向：把整个数据集当一个 batch
    logits = model(X).squeeze(-1)                                 # (N, 1) → (N,)
    # binary_cross_entropy_with_logits 比手写 sigmoid + BCE 数值稳定得多
    # —— 它内部用 softplus 形式 log(1 + e^{-|z|}) 改写，避免 e^{|z|} 上溢
    loss = F.binary_cross_entropy_with_logits(logits, y)

    # (b) 反向：清零梯度（绝不能漏！），再 backward
    optimizer.zero_grad()
    loss.backward()

    # (c) 更新：optimizer 按 SGD 公式 θ ← θ - lr * θ.grad 修改参数
    optimizer.step()

    # 记录 loss 与准确率，用于画曲线（注意用 detach()/no_grad 避免拉住计算图）
    losses.append(loss.item())
    with torch.no_grad():
        pred = (torch.sigmoid(logits) > 0.5).float()
        accs.append((pred == y).float().mean().item())

    if (epoch + 1) % 20 == 0:
        print(f'epoch {epoch+1:3d}  loss {loss.item():.4f}  acc {accs[-1]:.3f}')

In [ ]:
# ============================================================
# Cell 6: 画训练曲线 —— loss 应该单调下降，acc 单调上升
# ============================================================
# 如果 loss 不降，常见原因是漏调 zero_grad / lr 太大或太小 / target dtype 不对。
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(losses)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].set_title('Training loss')
axes[1].plot(accs)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy')
axes[1].set_title('Training accuracy')
axes[1].set_ylim(0.4, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 7: 画决策边界 —— 直观看 ReLU MLP 学到什么
# ============================================================
# 在 (x1, x2) 平面上铺一张细密网格，喂给训练好的模型，按预测概率上色。
import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_decision_boundary(model, X, y, title):
    # 在数据点的外接矩形 + 一点 padding 上铺网格
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    # (40000, 2) 的网格点
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():                          # 推理无需梯度
        probs = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)

    plt.figure(figsize=(5, 4))
    plt.contourf(xx, yy, probs, levels=20, cmap='RdBu_r', alpha=0.6)
    plt.contour(xx, yy, probs, levels=[0.5], colors='k', linewidths=1.5)
    plt.scatter(X[y == 0, 0], X[y == 0, 1], s=8, color='C0', edgecolor='w', linewidth=0.3)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], s=8, color='C3', edgecolor='w', linewidth=0.3)
    plt.title(title)
    plt.gca().set_aspect('equal')
    plt.show()

plot_decision_boundary(model, X.numpy(), y.numpy(),
                       title='MLP (ReLU) decision boundary: curved nonlinear')

In [ ]:
# ============================================================
# Cell 8: 反例 —— 去掉 ReLU 之后，多层退化成单层
# ============================================================
# 用同样的训练循环训一个"全线性"的 2 层网络，决策边界应当只能是一条直线。
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class LinearOnlyMLP(nn.Module):
    """故意去掉 ReLU —— 两层 Linear 叠起来等价于一层。"""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(self.fc1(x))            # 注意：没有 relu

linear_model = LinearOnlyMLP()
opt2 = torch.optim.SGD(linear_model.parameters(), lr=0.1)

for epoch in range(200):
    logits = linear_model(X).squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logits, y)
    opt2.zero_grad()
    loss.backward()
    opt2.step()

with torch.no_grad():
    pred = (torch.sigmoid(linear_model(X).squeeze(-1)) > 0.5).float()
    print(f'去掉 ReLU 后训练 200 epoch  →  acc = {(pred == y).float().mean().item():.3f}')
    print('（重点不是准确率——这个 SGD 配置下两边 acc 都在 ~0.86；')
    print(' 重点是「决策边界形态」：去掉 ReLU 后只能画直线，永远没法贴合两个半月的弯曲交界）')

plot_decision_boundary(linear_model, X.numpy(), y.numpy(),
                       title='No-ReLU MLP: decision boundary stays a straight line')

In [ ]:
# ============================================================
# Cell 9: 把训练循环再抽象一层 —— 这就是后续所有训练任务共用的骨架
# ============================================================
# 后续的语言模型预训练、SFT、LoRA、RLHF 等训练循环，与下面这几行的区别仅在于：
#   - model 从 MLP 换成 Transformer / LoRA-wrapped LLM
#   - loss 从 BCE 换成 cross_entropy / KL
#   - 优化器从 SGD 换成 AdamW + warmup-cosine 调度
#   - 多了 gradient_accumulation / grad_clipping / mixed_precision
# 但循环结构本身一字不改。把这套骨架记到肌肉记忆。

def train(model, optimizer, loss_fn, data, num_epochs):
    """通用训练循环 —— forward → backward → step。"""
    X, y = data
    history = []
    for epoch in range(num_epochs):
        # forward
        pred = model(X).squeeze(-1)
        loss = loss_fn(pred, y)

        # backward + step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.item())
    return history

# 演示一遍：换 AdamW，看是否更快收敛
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
model_adam = MLP()
opt_adam = torch.optim.AdamW(model_adam.parameters(), lr=1e-2)
hist_adam = train(model_adam, opt_adam, F.binary_cross_entropy_with_logits, (X, y), 200)

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.plot(losses, label='SGD lr=0.1')
plt.plot(hist_adam, label='AdamW lr=1e-2')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.legend(); plt.title('SGD vs AdamW: Adam reaches low loss faster')
plt.show()